In [2]:
# =========================
# Finance robustness test
# Replace PCA factors with selected raw financial variables
# =========================

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


# =========================
# 1. Load data
# =========================
FIN_RAW_PATH = "finance_features_ml_ready.csv"   # change path if needed
Y_PATH = "deepseek_Y_quantile_final.csv"

fin_raw = pd.read_csv(FIN_RAW_PATH)
y = pd.read_csv(Y_PATH)

print("fin_raw shape:", fin_raw.shape)
print("y shape:", y.shape)

print("\nfin_raw columns:")
print(fin_raw.columns.tolist())

print("\ny columns:")
print(y.columns.tolist())


# =========================
# 2. Select raw financial variables
# =========================
candidate_vars = [
    "rd_to_sales",
    "p_to_s",
    "mkt_to_book",
    "op_margin",
    "cash_ratio",
    "debt_to_assets",
    "log_mktcap",
]

# Keep only columns that actually exist
selected_vars = [c for c in candidate_vars if c in fin_raw.columns]

print("\nSelected raw finance variables:")
print(selected_vars)

if len(selected_vars) == 0:
    raise ValueError("None of the selected raw financial variables were found. Please check fin_raw.columns.")


# =========================
# 3. Prepare Y
# =========================
y_keep = y[["ticker", "label"]].drop_duplicates().copy()

label_map = {
    "Vulnerable": 1,
    "Resilient": 0
}
y_keep["label_binary"] = y_keep["label"].map(label_map)

print("\nY label counts:")
print(y_keep["label"].value_counts(dropna=False))


# =========================
# 4. Identify ticker column in finance raw
# =========================
# Common cases: tic or ticker
if "tic" in fin_raw.columns:
    fin_raw = fin_raw.rename(columns={"tic": "ticker"})
elif "ticker" not in fin_raw.columns:
    raise ValueError("Could not find ticker column in finance raw file. Expected 'tic' or 'ticker'.")

# Keep only required columns
keep_cols = ["ticker"] + selected_vars
if "gvkey" in fin_raw.columns:
    keep_cols.append("gvkey")
if "datadate" in fin_raw.columns:
    keep_cols.append("datadate")

fin_raw_sub = fin_raw[keep_cols].drop_duplicates().copy()


# =========================
# 5. Winsorize selected variables (1st / 99th pct)
# =========================
# This helps stabilize extreme ratio variables.
for col in selected_vars:
    lower = fin_raw_sub[col].quantile(0.01)
    upper = fin_raw_sub[col].quantile(0.99)
    fin_raw_sub[col] = fin_raw_sub[col].clip(lower, upper)

print("\nWinsorization completed.")


# =========================
# 6. Merge raw finance with Y
# =========================
model_fin_raw = fin_raw_sub.merge(
    y_keep,
    on="ticker",
    how="inner"
)

print("\nMerged raw-finance robustness dataset shape:", model_fin_raw.shape)
print("\nMerged label counts:")
print(model_fin_raw["label"].value_counts(dropna=False))

print("\nDuplicate tickers:", model_fin_raw["ticker"].duplicated().sum())
print("\nMissing values:")
print(model_fin_raw.isna().sum().sort_values(ascending=False).head(20))


# =========================
# 7. Define X and y
# =========================
X = model_fin_raw[selected_vars].copy()
y_binary = model_fin_raw["label_binary"].copy()

print("\nX shape:", X.shape)
print("y shape:", y_binary.shape)


# =========================
# 8. Define models
# =========================
def build_model(model_name):
    if model_name == "logit_l1":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="saga",
                C=1.0,
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            ))
        ])

    elif model_name == "logit_l2":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                C=1.0,
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            ))
        ])

    elif model_name == "random_forest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=42
            ))
        ])

    elif model_name == "xgboost":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                random_state=42,
                eval_metric="logloss"
            ))
        ])

    else:
        raise ValueError(f"Unknown model_name: {model_name}")


# =========================
# 9. Evaluation function
# =========================
def evaluate_model(X, y, model_name):
    model = build_model(model_name)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "accuracy": "accuracy",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    return {
        "feature_set": "X_fin_raw_selected",
        "model_name": model_name,
        "n_samples": len(X),
        "n_features": X.shape[1],
        "accuracy_mean": np.mean(scores["test_accuracy"]),
        "accuracy_std": np.std(scores["test_accuracy"]),
        "f1_mean": np.mean(scores["test_f1"]),
        "f1_std": np.std(scores["test_f1"]),
        "roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "roc_auc_std": np.std(scores["test_roc_auc"]),
    }


# =========================
# 10. Run robustness models
# =========================
model_names = ["logit_l1", "logit_l2", "random_forest", "xgboost"]

results = []
for m in model_names:
    print(f"Running {m} ...")
    results.append(evaluate_model(X, y_binary, m))

results_df = pd.DataFrame(results)

print("\n=== Finance robustness results (raw selected variables) ===")
print(results_df.round(4).to_string(index=False))


# =========================
# 11. Save outputs
# =========================
model_fin_raw.to_csv("model_finance_raw_selected.csv", index=False)
results_df.to_csv("finance_robustness_results.csv", index=False)

print("\nSaved files:")
print("  model_finance_raw_selected.csv")
print("  finance_robustness_results.csv")

fin_raw shape: (403, 36)
y shape: (158, 17)

fin_raw columns:
['tic', 'gvkey', 'datadate', 'assets', 'revenue', 'net_income', 'operating_income', 'cash', 'debt_total', 'equity', 'rd', 'mktcap', 'cogs', 'intangibles', 'current_assets', 'current_liabilities', 'long_term_debt', 'roa', 'roe', 'op_margin', 'net_margin', 'cash_to_assets', 'debt_to_assets', 'debt_to_equity', 'current_ratio', 'rd_to_sales', 'log_assets', 'log_mktcap', 'mkt_to_book', 'p_to_s', 'p_to_e', 'gross_margin', 'asset_turnover', 'intangibles_to_assets', 'cash_ratio', 'long_term_debt_ratio']

y columns:
['company_id', 'cik', 'ticker', 'company_name', 'sic', 'sic_description', 'start_date_actual', 'start_price', 'end_date_actual', 'end_price', 'raw_return', 'normal_return_spy', 'excess_return', 'CAR', 'MaxDrawdown', 'Volatility', 'label']

Selected raw finance variables:
['rd_to_sales', 'p_to_s', 'mkt_to_book', 'op_margin', 'cash_ratio', 'debt_to_assets', 'log_mktcap']

Y label counts:
label
Vulnerable    79
Resilient    

In [3]:
# =========================
# Finance robustness test + industry controls
# Use selected raw financial variables
# Add broad industry controls based on SIC description
# =========================

import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


# =========================
# 1. Load data
# =========================
FIN_RAW_PATH = "finance_features_ml_ready.csv"   # change path if needed
Y_PATH = "deepseek_Y_quantile_final.csv"

fin_raw = pd.read_csv(FIN_RAW_PATH)
y = pd.read_csv(Y_PATH)

print("fin_raw shape:", fin_raw.shape)
print("y shape:", y.shape)

print("\nfin_raw columns:")
print(fin_raw.columns.tolist())

print("\ny columns:")
print(y.columns.tolist())


# =========================
# 2. Select raw financial variables
# =========================
candidate_vars = [
    "rd_to_sales",
    "p_to_s",
    "mkt_to_book",
    "op_margin",
    "cash_ratio",
    "debt_to_assets",
    "log_mktcap",
]

selected_vars = [c for c in candidate_vars if c in fin_raw.columns]

print("\nSelected raw finance variables:")
print(selected_vars)

if len(selected_vars) == 0:
    raise ValueError("None of the selected raw financial variables were found.")


# =========================
# 3. Prepare Y + industry info
# =========================
# Keep label and SIC description
y_keep = y[["ticker", "label", "sic_description"]].drop_duplicates().copy()

label_map = {
    "Vulnerable": 1,
    "Resilient": 0
}
y_keep["label_binary"] = y_keep["label"].map(label_map)

print("\nY label counts:")
print(y_keep["label"].value_counts(dropna=False))


# =========================
# 4. Create broad industry categories
# =========================
def map_broad_sector(s):
    if pd.isna(s):
        return "other"

    s = str(s).lower()

    # Semiconductor / chips / electronics
    if any(k in s for k in [
        "semiconductor", "electronic component", "electronics",
        "chip", "integrated circuit"
    ]):
        return "semiconductor"

    # Software / SaaS / cybersecurity / IT services
    elif any(k in s for k in [
        "software", "prepackaged software", "programming",
        "computer systems design", "data processing",
        "computer integrated systems", "it service", "cyber"
    ]):
        return "software"

    # Internet / platform / media / communications
    elif any(k in s for k in [
        "internet", "online", "portal", "streaming",
        "broadcast", "telecommunication", "communications",
        "advertising"
    ]):
        return "internet"

    # Energy / power / utility
    elif any(k in s for k in [
        "energy", "power", "utility", "utilities",
        "electric", "oil", "gas"
    ]):
        return "energy"

    else:
        return "other"


y_keep["broad_sector"] = y_keep["sic_description"].apply(map_broad_sector)

print("\nBroad sector counts:")
print(y_keep["broad_sector"].value_counts(dropna=False))


# =========================
# 5. Create industry dummies
# =========================
# Drop first to avoid perfect multicollinearity for linear models
sector_dummies = pd.get_dummies(
    y_keep["broad_sector"],
    prefix="sector",
    drop_first=True,
    dtype=int
)

y_keep = pd.concat([y_keep, sector_dummies], axis=1)

industry_cols = sector_dummies.columns.tolist()

print("\nIndustry control columns:")
print(industry_cols)


# =========================
# 6. Prepare finance raw table
# =========================
if "tic" in fin_raw.columns:
    fin_raw = fin_raw.rename(columns={"tic": "ticker"})
elif "ticker" not in fin_raw.columns:
    raise ValueError("Could not find ticker column in finance raw file. Expected 'tic' or 'ticker'.")

keep_cols = ["ticker"] + selected_vars
if "gvkey" in fin_raw.columns:
    keep_cols.append("gvkey")
if "datadate" in fin_raw.columns:
    keep_cols.append("datadate")

fin_raw_sub = fin_raw[keep_cols].drop_duplicates().copy()


# =========================
# 7. Winsorize raw finance variables
# =========================
for col in selected_vars:
    lower = fin_raw_sub[col].quantile(0.01)
    upper = fin_raw_sub[col].quantile(0.99)
    fin_raw_sub[col] = fin_raw_sub[col].clip(lower, upper)

print("\nWinsorization completed.")


# =========================
# 8. Merge finance + Y + industry controls
# =========================
merge_cols = ["ticker", "label", "label_binary", "sic_description", "broad_sector"] + industry_cols

model_fin_ind = fin_raw_sub.merge(
    y_keep[merge_cols],
    on="ticker",
    how="inner"
)

print("\nMerged finance + industry-controls dataset shape:", model_fin_ind.shape)
print("\nMerged label counts:")
print(model_fin_ind["label"].value_counts(dropna=False))

print("\nDuplicate tickers:", model_fin_ind["ticker"].duplicated().sum())
print("\nMissing values:")
print(model_fin_ind.isna().sum().sort_values(ascending=False).head(20))


# =========================
# 9. Define X and y
# =========================
feature_cols = selected_vars + industry_cols

X = model_fin_ind[feature_cols].copy()
y_binary = model_fin_ind["label_binary"].copy()

print("\nFinal feature columns:")
print(feature_cols)

print("\nX shape:", X.shape)
print("y shape:", y_binary.shape)


# =========================
# 10. Define models
# =========================
def build_model(model_name):
    if model_name == "logit_l1":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l1",
                solver="saga",
                C=1.0,
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            ))
        ])

    elif model_name == "logit_l2":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                penalty="l2",
                solver="lbfgs",
                C=1.0,
                max_iter=5000,
                class_weight="balanced",
                random_state=42
            ))
        ])

    elif model_name == "random_forest":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_leaf=3,
                class_weight="balanced",
                random_state=42
            ))
        ])

    elif model_name == "xgboost":
        return Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=300,
                max_depth=3,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                reg_lambda=1.0,
                random_state=42,
                eval_metric="logloss"
            ))
        ])

    else:
        raise ValueError(f"Unknown model_name: {model_name}")


# =========================
# 11. Evaluation function
# =========================
def evaluate_model(X, y, model_name):
    model = build_model(model_name)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    scoring = {
        "accuracy": "accuracy",
        "f1": "f1",
        "roc_auc": "roc_auc"
    }

    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        return_train_score=False
    )

    return {
        "feature_set": "X_fin_raw_selected_plus_industry",
        "model_name": model_name,
        "n_samples": len(X),
        "n_features": X.shape[1],
        "accuracy_mean": np.mean(scores["test_accuracy"]),
        "accuracy_std": np.std(scores["test_accuracy"]),
        "f1_mean": np.mean(scores["test_f1"]),
        "f1_std": np.std(scores["test_f1"]),
        "roc_auc_mean": np.mean(scores["test_roc_auc"]),
        "roc_auc_std": np.std(scores["test_roc_auc"]),
    }


# =========================
# 12. Run models
# =========================
model_names = ["logit_l1", "logit_l2", "random_forest", "xgboost"]

results = []
for m in model_names:
    print(f"Running {m} ...")
    results.append(evaluate_model(X, y_binary, m))

results_df = pd.DataFrame(results)

print("\n=== Finance robustness results + industry controls ===")
print(results_df.round(4).to_string(index=False))


# =========================
# 13. Save outputs
# =========================
model_fin_ind.to_csv("model_finance_raw_selected_plus_industry.csv", index=False)
results_df.to_csv("finance_robustness_industry_controls_results.csv", index=False)

print("\nSaved files:")
print("  model_finance_raw_selected_plus_industry.csv")
print("  finance_robustness_industry_controls_results.csv")

fin_raw shape: (403, 36)
y shape: (158, 17)

fin_raw columns:
['tic', 'gvkey', 'datadate', 'assets', 'revenue', 'net_income', 'operating_income', 'cash', 'debt_total', 'equity', 'rd', 'mktcap', 'cogs', 'intangibles', 'current_assets', 'current_liabilities', 'long_term_debt', 'roa', 'roe', 'op_margin', 'net_margin', 'cash_to_assets', 'debt_to_assets', 'debt_to_equity', 'current_ratio', 'rd_to_sales', 'log_assets', 'log_mktcap', 'mkt_to_book', 'p_to_s', 'p_to_e', 'gross_margin', 'asset_turnover', 'intangibles_to_assets', 'cash_ratio', 'long_term_debt_ratio']

y columns:
['company_id', 'cik', 'ticker', 'company_name', 'sic', 'sic_description', 'start_date_actual', 'start_price', 'end_date_actual', 'end_price', 'raw_return', 'normal_return_spy', 'excess_return', 'CAR', 'MaxDrawdown', 'Volatility', 'label']

Selected raw finance variables:
['rd_to_sales', 'p_to_s', 'mkt_to_book', 'op_margin', 'cash_ratio', 'debt_to_assets', 'log_mktcap']

Y label counts:
label
Vulnerable    79
Resilient    